In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# !pip install torch==2.4.1+cu121 torchvision==0.19.1+cu121 \
#     --extra-index-url https://download.pytorch.org/whl/cu121

# !pip install torch-scatter -f https://data.pyg.org/whl/torch-2.4.1+cu121.html

# !pip install python-box>=7.0.0 \
#              lightning>=2.6.1 \
#              pypose>=0.7.0 \
#              ipython>=8.10.0 \
#              numpy>=1.24.0 \
#              opencv-python>=4.8.0 \
#              scipy>=1.10.0 \
#              matplotlib>=3.5.0 \
#              pandas>=2.0.0 \
#              PyYAML>=6.0 \
             
# !pip install wandb

In [ ]:
import os
import sys

import numpy as np 
import torch

import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.callbacks import LearningRateMonitor, RichProgressBar, RichModelSummary, DeviceStatsMonitor
from lightning.pytorch.loggers import WandbLogger

from box import Box
import yaml

from pathlib import Path
import time 

## Settings 
___

In [ ]:
BASE_DIR = Path.cwd().parent
VERSION = 1

# -- for colab traning
# BASE_DIR = '/content/drive/MyDrive/SonarOdometry'
# from google.colab import drive
# drive.mount('/content/drive')


if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print(f'Setting root dir to: {BASE_DIR}')


# --- if colab
# copy traning data to
# !cp /content/drive/MyDrive/SonarOdometry/data/dataset.zip /content/
# !unzip -q /content/dataset.zip -d /content/dataset/
# data_dir = '/content/dataset/'


# import modules
from src.models.dpso_train import DPSO_train as DPSO
from src.data_loader.lightning_module import DPSO_LightningModule
from src.data_loader.data_module_lightning import SonarSimDataModule
from src.data_loader.transforms import SonarDatasetTranforms


# model configuration files
model_config_pth = os.path.join(BASE_DIR, 'config', 'model.yaml')
sonar_config_pth = os.path.join(BASE_DIR, 'config', 'sonar.yaml')

# dataset path
data_dir = os.path.join(BASE_DIR, 'data')

# output data 
output_dir = os.path.join(BASE_DIR, 'training/output/run_{VERSION}}') 
os.makedirs(output_dir, exist_ok=True)

print(f'Training results directory: {output_dir}')

Setting root dir to: c:\Users\janis\Projekty\Magisterka\SonarOdometry
Training results directory: c:\Users\janis\Projekty\Magisterka\SonarOdometry\training/output/run_test


## Dataset configuration
___

In [ ]:
# === DATASET CONFIGURATION ===
with open(model_config_pth, "r") as f:
            model_config = Box(yaml.safe_load(f))

fls_img_size = (model_config.FLS_INPUT_HEIGHT, model_config.FLS_INPUT_WIDTH)
transforms = SonarDatasetTranforms

## **Supervised learning** 
### Traning configuration
___

In [ ]:
# === TRAINING CONFIGURATION ===

# dataloader: 
epochs = 4
batch_size = 1
num_workers = 2
val_interval = 500
frames_in_series = 12
init_frames = model_config.INIT_FRAMES # init_frames < frames_in_series

mode = 'supervised'

supervised_traning_param = {
    'freeze_poses_steps':1000,
    'init_pose_max_noise':0.2,
    'loss_weight_trans':1.0,
    'loss_weight_rot':1.0,
    'loss_weight_proj_r':0.1,
    'loss_weight_proj_theta':1.0,
    'loss_weight_weights':0.2,
    'val_interval': val_interval
}
traning_param = supervised_traning_param

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# init model 
dpso = DPSO(model_config_pth, sonar_config_pth, batch_size, frames_in_series, init_frames)
# init lighning wrapper
model = DPSO_LightningModule(dpso, mode, traning_param)
# init data module 
data_dir_supervies = os.path.join(data_dir, 'supervised')
dm = SonarSimDataModule(data_dir_supervies, batch_size, num_workers, transforms, frames_in_series)

c:\Users\janis\miniconda3\envs\so\lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.


In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=output_dir,          
    filename='dpso-supervised_ver1-{step:02d}-{val_loss:.4f}', 
    monitor='val_loss',              
    mode='min',                      
    save_top_k=5,                    
    save_last=True                   
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

wandb_logger = WandbLogger(
    project="SonarOdometry",  # Project name
    entity="mjanis-agh",
    name="dpso_v1_test_supervised",      # Run name
    log_model="all"           # Save all checkpoints on wandb cloud
)

# early_stop_callback = EarlyStopping(
#     monitor="val_loss", 
#     patience=7,          # Ile epok czekać na poprawę, zanim przerwie
#     mode="min"
# )

# callbacks = [checkpoint_callback, lr_monitor, DeviceStatsMonitor()]

callbacks = [checkpoint_callback, lr_monitor, #, early_stop_callback,
             DeviceStatsMonitor(), RichProgressBar(), RichModelSummary(max_depth=2)]

trainer = pl.Trainer(
    logger = wandb_logger,
    callbacks = callbacks,
    accumulate_grad_batches=4,
    # gradient_clip_val=1.0,
    max_epochs=epochs,             
    accelerator="auto",          
    devices="auto",           
    log_every_n_steps=5, 
    val_check_interval=500, 
    # precision="16-mixed",
    profiler="simple"
)



GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model, datamodule=dm)

best_ckpt_path_supervised = checkpoint_callback.best_model_path
best_score_supervised = checkpoint_callback.best_model_score


  | Name  | Type       | Params | Mode | FLOPs
----------------------------------------------------
0 | model | DPSO_train | 3.3 M  | eval | 0    
----------------------------------------------------
3.3 M     Trainable params
0         Non-trainable params
3.3 M     Total params
13.283    Total estimated model params size (MB)
0         Modules in train mode
112       Modules in eval mode
0         Total Flops


c:\Users\janis\miniconda3\envs\so\lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\janis\miniconda3\envs\so\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


c:\Users\janis\miniconda3\envs\so\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.
c:\Users\janis\miniconda3\envs\so\lib\site-packages\lightning\pytorch\loops\fit_loop.py:534: Found 113 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Epoch 0:   0%|          | 0/21 [00:00<?, ?it/s]

: 

## Self-supervised learning
### Traning configuration
___


In [ ]:
# === TRAINING CONFIGURATION ===

# dataloader: 
epochs = 4
batch_size = 1
num_workers = 2

frames_in_series = 12
init_frames = model_config.INIT_FRAMES # init_frames < frames_in_series

mode = 'self-supervised'

selfsupervised_traning_param = {
    'freeze_poses_steps':0,
    'freeze_delta_loss_step':500,
    'init_pose_max_noise':0.2,
    # 'loss_weight_trans':1.0, 
    # 'loss_weight_rot':1.0,
    'loss_weight_weights':0.2,
    'loss_weight_proj_r':0.1,
    'loss_weight_proj_theta':1.0,
    'val_interval': val_interval
}

traning_param = selfsupervised_traning_param

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# init model 
dpso = DPSO(model_config_pth, sonar_config_pth, batch_size, frames_in_series, init_frames)

model = DPSO_LightningModule.load_from_checkpoint(
    checkpoint_path=best_ckpt_path_supervised,
    model=dpso,
    mode=mode,
    traning_param=traning_param) 

# init data module 
data_dir_selfsupervies = os.path.join(data_dir) #, 'selfsupervised')
dm = SonarSimDataModule(data_dir_selfsupervies, batch_size, num_workers, transforms, frames_in_series)

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=output_dir,          
    filename='dpso-selfsupervised_ver1-{step:02d}-{val_loss:.4f}', 
    monitor='val_loss',              
    mode='min',                      
    save_top_k=5,                    
    save_last=True                   
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

wandb_logger = WandbLogger(
    project="SonarOdometry",
    entity="mjanis-agh",  
    name="dpso_v1_test_selfsupervised",      
    log_model="all"           
)

# early_stop_callback = EarlyStopping(
#     monitor="val_loss", 
#     patience=7,          
#     mode="min"
# )


# callbacks = [checkpoint_callback, lr_monitor, DeviceStatsMonitor()]

callbacks = [checkpoint_callback, lr_monitor,#, early_stop_callback,
             DeviceStatsMonitor(), RichProgressBar(), RichModelSummary(max_depth=2)]

trainer = pl.Trainer(
    logger=wandb_logger,
    callbacks = callbacks,
    accumulate_grad_batches=4,
    # gradient_clip_val=1.0,
    max_epochs=epochs,             
    accelerator="auto",          
    devices="auto",            
    log_every_n_steps=5,
    val_check_interval=500,
    # precision="16-mixed",
    profiler="simple"
)

In [ ]:
trainer.fit(model, datamodule=dm)

best_ckpt_path_selfsupervised = checkpoint_callback.best_model_path
best_score_selfsupervised = checkpoint_callback.best_model_score